# DDPM Linear Beta Schedule: beta_t, alpha_t, alpha_bar_t

The forward (noising) process of DDPM (Ho et al., 2020, Sec. 2) corrupts a
clean image $x_0$ step by step, for $t = 1, \dots, T$:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\big(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\big)$$

$\beta_t \in (0,1)$ is a fixed (not learned) **variance schedule**, a
hyperparameter chosen in advance. The paper's "linear schedule" sets
$\beta_t$ to increase linearly from $\beta_1$ to $\beta_T$ (paper
defaults: $\beta_1 = 10^{-4}$, $\beta_T = 0.02$, $T = 1000$).

Two quantities are derived from $\beta_t$ and used throughout DDPM:

| symbol | definition | meaning |
|---|---|---|
| $\beta_t$ | linearly spaced, $\beta_1 \to \beta_T$ | variance added to the signal at step $t$ |
| $\alpha_t$ | $1 - \beta_t$ | fraction of signal **kept** at step $t$ (per-step) |
| $\bar\alpha_t$ | $\prod_{s=1}^{t} \alpha_s$ | fraction of signal **kept cumulatively** from $x_0$ to $x_t$ |

$\bar\alpha_t$ is what later gives the closed-form one-shot noising
formula $q(x_t \mid x_0) = \mathcal{N}(x_t; \sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t) I)$
(not implemented in this notebook -- this notebook only builds and
visualizes the schedule itself).

**Indexing convention.** The paper indexes $t = 1, \dots, T$. The code
below returns plain 0-indexed tensors of length $T$ (array index $0$
$\leftrightarrow$ paper's $t=1$, array index $T-1$ $\leftrightarrow$
paper's $t=T$); printed tables convert back to the paper's 1-indexed
$t$ for readability.

Core code lives in `code/beta_schedule.py` (`linear_beta_schedule`,
`compute_alpha_and_alpha_bar`, `schedule_table`,
`plot_beta_alpha_alphabar`), version-controlled and unit-checked
(`python code/beta_schedule.py --smoke-test`); this notebook is a thin
orchestration layer that calls it. No model, no dataset, no GPU needed
-- $\beta_t$, $\alpha_t$, $\bar\alpha_t$ are a closed-form 1-D schedule.

## 1. Clone/pull repository

Schedule code lives in `code/beta_schedule.py`, not duplicated in this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 2. Choose schedule configuration and compute beta_t, alpha_t, alpha_bar_t

`T = 1000`, `beta_start = 1e-4`, `beta_end = 0.02` are the DDPM paper's own linear-schedule defaults (Ho et al., 2020, Sec. 4).

In [ ]:
from beta_schedule import linear_beta_schedule, compute_alpha_and_alpha_bar, schedule_table, plot_beta_alpha_alphabar

NUM_TIMESTEPS = 1000
BETA_START = 1e-4  # beta_1 in the paper
BETA_END = 0.02    # beta_T in the paper

betas = linear_beta_schedule(NUM_TIMESTEPS, BETA_START, BETA_END)   # [T]
alphas, alpha_bars = compute_alpha_and_alpha_bar(betas)              # [T], [T]

print(f"betas.shape      = {tuple(betas.shape)}")
print(f"alphas.shape     = {tuple(alphas.shape)}")
print(f"alpha_bars.shape = {tuple(alpha_bars.shape)}")


## 3. Numeric table

10 evenly spaced timesteps (including $t=1$ and $t=T$), paper's
1-indexed $t$. Read down the columns:
- $\beta_t$ grows linearly from $10^{-4}$ to $0.02$ -- a small number throughout, by design (each single step adds only a little noise).
- $\alpha_t = 1-\beta_t$ mirrors $\beta_t$: it shrinks linearly but stays close to $1$ (each step keeps almost all of the previous step's signal).
- $\bar\alpha_t$ is the **product** of a thousand such $\alpha_t$ factors -- even though every individual $\alpha_t \approx 0.98$–$1.0$, multiplying $1000$ of them compounds into a value that collapses from $\approx 1$ down to $\approx 4\times10^{-5}$ by $t=T$.

In [ ]:
print(schedule_table(betas, alphas, alpha_bars, num_rows=10))


## 4. Graphical visualization

Three panels, one per quantity, all against $t$:
1. $\beta_t$ -- a straight line by construction (that's the "linear" in "linear schedule").
2. $\alpha_t = 1-\beta_t$ -- also a straight line, the mirror image of panel 1, but with a much smaller visual range (note the y-axis: it only spans $[0.98, 1.0]$).
3. $\bar\alpha_t$ -- plotted on a **log y-axis**. Two linear-looking per-step quantities (panels 1-2) combine, through the cumulative product, into a curve that is close to a straight line *in log space*: $\log\bar\alpha_t = \sum_{s\le t} \log\alpha_s \approx \sum_{s\le t} (-\beta_s)$ for small $\beta_s$ (since $\log(1-\beta)\approx-\beta$), i.e. $\bar\alpha_t$ decays roughly like $\exp(-\sum_{s\le t}\beta_s)$ -- compounding many small per-step losses into a large cumulative one, the same arithmetic as compound interest.

In [ ]:
plot_beta_alpha_alphabar(betas, alphas, alpha_bars)


## 5. Interpretation

- At $t=1$: $\bar\alpha_1 = \alpha_1 \approx 0.9999$ -- $x_1$ is almost indistinguishable from $x_0$ (one tiny noise step).
- At $t=T=1000$: $\bar\alpha_T \approx 4\times10^{-5} \approx 0$ -- by the closed-form relation $q(x_T\mid x_0)=\mathcal{N}(x_T;\sqrt{\bar\alpha_T}x_0,(1-\bar\alpha_T)I)$, this means $x_T$ retains almost none of $x_0$'s signal ($\sqrt{\bar\alpha_T}\approx 0.0064$) and is essentially pure Gaussian noise ($1-\bar\alpha_T\approx 1$) -- exactly the reverse-process starting distribution DDPM assumes.
- $\bar\alpha_t$ is **monotonically decreasing** (verified in `code/beta_schedule.py`'s smoke test) because it is a running product of factors $\alpha_t\in(0,1)$: every extra step can only remove more signal, never restore it.
- This is why $T$ and the $(\beta_{\text{start}},\beta_{\text{end}})$ endpoints matter: they jointly control how quickly $\bar\alpha_t$ collapses to $\approx 0$, i.e. how many forward steps are "used up" before the signal is fully destroyed.